In [41]:
import pandas as pd
import numpy as np
import openmeteo_requests
import numpy as np
from openmeteo_sdk.Variable import Variable

openmeteo = openmeteo_requests.Client()

url = "https://api.open-meteo.com/v1/forecast"


def elevation(lat: float, lon: float) -> float:
    '''
    Takes an elevation and a temperature and returns the air density
    of the environment (on earth).
    '''

    params = {
        "latitude": lat,
        "longitude": lon,
        "hourly": ["temperature_2m", "relative_humidity_2m"]
    }

    responses = openmeteo.weather_api(url, params=params)
    response = responses[0]

    e = response.Elevation()
    return e

def temp(lat, lon):

    params = {
        "latitude": lat,
        "longitude": lon,
        "hourly": ["temperature_2m", "relative_humidity_2m"]
    }

    responses = openmeteo.weather_api(url, params=params)
    response = responses[0]

    e = response.Elevation()

    hourly = response.Hourly()
    hourly_variables = list(map(lambda i: hourly.Variables(i), range(0, hourly.VariablesLength())))

    hourly_temperature_2m = next(
        filter(
            lambda x: x.Variable() == Variable.temperature and x.Altitude() == 2,
            hourly_variables
        )
    ).ValuesAsNumpy()

    hourly_data = {"date": pd.date_range(
        start = pd.to_datetime(hourly.Time(), unit = "s"),
        end = pd.to_datetime(hourly.TimeEnd(), unit = "s"),
        freq = pd.Timedelta(seconds = hourly.Interval()),
        inclusive = "left"
    )}

    hourly_data["temperature_2m"] = hourly_temperature_2m
    hourly_dataframe_pd = pd.DataFrame(data = hourly_data)

    df = hourly_dataframe_pd.groupby('date')['temperature_2m'].mean()
    return df

In [43]:
e = elevation(47.5914, -122.3325)
print("Seattle elevation:", e)
T = temp(47.5914, -122.3325)
T
# df = T.groupby('date')['temperature_2m'].mean()
df['2026-08-05 00:00:00']

Seattle elevation: 14.0


25.040499

In [ ]:
e = 14.0  # m
R = 8.314462  # J/(mol*K)
P_0 = 101325.0  # Pa
T = 293.15 # K
year = 2025

P = P_0 * (1 - 2.25577 * 10 ** (-5) * e) ** 5.25588
rho_mol = P / (R * T) # mol / m^3

kg_mol = 0.0289647  # kg / mol
print(rho_mol * kg_mol)

1.2021000609287495
